# DK-SOFNN: Grid-81 Initialisation
## Combined Cycle Power Plant (CCPP) Dataset

Both the **Source FNN** (1 500 samples) and the **Target FNN** (100 samples) start
with **81 rules** produced by a 3-level linguistic grid over the 4 CCPP inputs
(3⁴ = 81 IF-THEN rules covering the full normalised input space).

The paper's grow / prune / replace structure-adjustment mechanism then runs on
each network, producing the graphs below.

**Graphs produced**
1. Source FNN — rule count evolution over epochs (starts at K = 81)
2. Target DK-SOFNN — rule count evolution over epochs (starts at K = 81)


In [ ]:
# Cell 1 – Setup
!pip install openpyxl -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import MaxNLocator
from itertools import product as iterproduct
from copy import deepcopy
import warnings, os, pickle, time
warnings.filterwarnings('ignore')

np.random.seed(42)

plt.rcParams.update({
    'font.size': 10,
    'axes.titlesize': 11,
    'axes.labelsize': 10,
    'lines.linewidth': 1.5,
    'grid.alpha': 0.3,
    'figure.dpi': 100,
})

EPS     = 1e-8
EPS_IDX = 1e-10

print('Setup complete.')


In [ ]:
# Cell 2 – Data Loading & Preprocessing (same as base notebook)
def load_ccpp(filepath='Folds5x2_pp.xlsx'):
    try:
        df = pd.read_excel(filepath, sheet_name='Sheet1')
    except Exception:
        df = pd.read_excel(filepath)
    return df.values.astype(np.float64)


def prepare_ccpp(data, n_src=1500, n_tgt_tr=100, n_tgt_te=300,
                 noise_mu=50, noise_sigma=10, seed=42):
    np.random.seed(seed)
    idx  = np.random.permutation(len(data))
    data = data[idx]
    X, y = data[:, :-1], data[:, -1]

    Xs  = X[:n_src];                ys  = y[:n_src]
    Xtt = X[n_src:n_src+n_tgt_tr]
    ytt = y[n_src:n_src+n_tgt_tr] + np.random.normal(noise_mu, noise_sigma, n_tgt_tr)
    Xte = X[n_src+n_tgt_tr:n_src+n_tgt_tr+n_tgt_te]
    yte = y[n_src+n_tgt_tr:n_src+n_tgt_tr+n_tgt_te]

    Xmin, Xmax = Xs.min(0), Xs.max(0)
    def norm_X(Z): return (Z - Xmin) / (Xmax - Xmin + 1e-8)

    ymin, ymax = ys.min(), ys.max()
    def norm_y(v): return (v - ymin) / (ymax - ymin + 1e-8)

    return {
        'Xs':  norm_X(Xs),  'ys':  norm_y(ys),
        'Xtt': norm_X(Xtt), 'ytt': norm_y(ytt),
        'Xte': norm_X(Xte), 'yte': norm_y(yte),
        'ymin': ymin, 'ymax': ymax
    }


raw = load_ccpp('Folds5x2_pp.xlsx')
DS  = prepare_ccpp(raw)
print(f'Split → Source: {DS["Xs"].shape[0]}  '
      f'Target-train: {DS["Xtt"].shape[0]}  '
      f'Target-test: {DS["Xte"].shape[0]}')


In [ ]:
# Cell 3 – RBF Fuzzy Neural Network  (updated: supports K=0 initialisation)
class RBFFNN:
    def __init__(self, n_in, n_rules=0, lr=0.1):
        self.P  = n_in
        self.K  = n_rules
        self.lr = lr
        if n_rules > 0:
            self.C = np.random.rand(n_rules, n_in)
            self.S = np.full((n_rules, n_in), 0.3)
            self.W = np.random.randn(n_rules) * 0.1
        else:
            self.C = np.empty((0, n_in))
            self.S = np.empty((0, n_in))
            self.W = np.empty(0)

    def forward(self, x):
        if self.K == 0:
            return 0.0, np.empty(0), np.empty(0)
        diff = x[None, :] - self.C
        u    = np.exp(-0.5 * ((diff / (self.S + EPS)) ** 2).sum(1))
        v    = u / (u.sum() + EPS_IDX)
        y    = float(self.W @ v)
        return y, u, v

    def predict(self, X):
        return np.array([self.forward(x)[0] for x in X])

    def update(self, x, y, yd, u, v,
               alpha=1.0, beta=0.0, Ws=None, Cs=None, Ss=None):
        if self.K == 0:
            return
        e  = y - yd
        gW = alpha * e * v
        if beta > 0 and Ws is not None:
            n = min(self.K, len(Ws))
            gW[:n] += beta * (self.W[:n] - Ws[:n])

        gC = np.zeros_like(self.C)
        gS = np.zeros_like(self.S)
        for b in range(self.K):
            s2 = self.S[b] ** 2 + EPS
            s3 = self.S[b] ** 3 + EPS
            Dc = v[b] * (self.W[b] - y) * (x - self.C[b]) / s2
            Ds = v[b] * (self.W[b] - y) * (x - self.C[b]) ** 2 / s3
            gC[b] = alpha * e * Dc
            gS[b] = alpha * e * Ds
            if beta > 0 and Cs is not None:
                kb = min(b, len(Cs) - 1)
                gC[b] += beta * (self.C[b] - Cs[kb])
                gS[b] += beta * (self.S[b] - Ss[kb])

        self.W -= self.lr * gW
        self.C -= self.lr * gC
        self.S  = np.maximum(self.S - self.lr * gS, 0.01)

    def add_rule(self, x, l=None):
        new_c = x.copy()
        if self.K == 0 or l is None:
            new_s = np.full(self.P, 0.3)
            new_w = float(np.random.randn() * 0.1)
        else:
            new_s = np.abs(x - self.C[l]) / 2.0 + 0.1
            new_w = float(self.W[l])
        self.C = np.vstack([self.C, new_c])
        self.S = np.vstack([self.S, new_s])
        self.W = np.append(self.W, new_w)
        self.K += 1

    def del_rule(self, l):
        if self.K <= 1:
            return
        self.C = np.delete(self.C, l, 0)
        self.S = np.delete(self.S, l, 0)
        self.W = np.delete(self.W, l)
        self.K -= 1

    def replace_rule(self, l_tgt, c_src, s_src, w_src):
        self.C[l_tgt] = c_src
        self.S[l_tgt] = s_src
        self.W[l_tgt] = w_src

    def clone(self):
        f   = RBFFNN(self.P, 0, self.lr)
        f.K = self.K
        f.C = self.C.copy()
        f.S = self.S.copy()
        f.W = self.W.copy()
        return f

print('RBFFNN defined (K=0 support enabled).')


In [ ]:
# Cell 4 – Fuzzy Rule Quality Indices (Eqs. 10-12, unchanged)
def compute_indices(U, y_arr):
    N, K  = U.shape
    eps   = EPS_IDX
    if N < 3:
        return np.ones(K), np.ones(K)/K, np.ones(K)/K

    total_u = U.sum(1)

    R = np.zeros(K)
    for l in range(K):
        a = U[:, l] - U[:, l].mean()
        b = total_u  - total_u.mean()
        denom = np.linalg.norm(a) * np.linalg.norm(b) + eps
        R[l]  = 1.0 / (abs(float(a @ b) / denom) + eps)

    M = U.sum(0) / (U.sum() + eps)

    C = np.zeros(K)
    for l in range(K):
        a = U[:, l] - y_arr
        b = total_u  - y_arr
        denom = np.linalg.norm(a) * np.linalg.norm(b) + eps
        d_l   = float(a @ b) / denom
        C[l]  = 1.0 / (abs(d_l) + eps)

    return R, M, C

print('Index functions defined.')


In [ ]:
# Cell 5 – Linguistic Grid Initialisation (3^4 = 81 rules)
# ──────────────────────────────────────────────────────────
# CCPP has 4 inputs (AT, V, AP, RH).  Each is split into 3 levels:
#   Low    centre = 1/6  ≈ 0.167
#   Medium centre = 3/6  = 0.500
#   High   centre = 5/6  ≈ 0.833
# sigma for each level = 1/6 (so adjacent centres are ~1 σ apart).

N_LEVELS  = 3
N_INPUTS  = 4   # CCPP inputs
LEVEL_CTR = np.array([1/6, 3/6, 5/6])   # Low / Medium / High centres
LEVEL_SIG = 1.0 / (N_LEVELS * 2)        # ≈ 0.167

def make_grid_81(n_inputs=N_INPUTS):
    """Return (centers, sigmas) arrays for the 3^n_inputs grid."""
    from itertools import product as iterproduct
    combos  = list(iterproduct(range(N_LEVELS), repeat=n_inputs))
    n_rules = len(combos)                        # 81
    centers = np.array([[LEVEL_CTR[lvl] for lvl in combo]
                        for combo in combos], dtype=float)
    sigmas  = np.full((n_rules, n_inputs), LEVEL_SIG)
    return centers, sigmas

grid_centers, grid_sigmas = make_grid_81()
print(f'Linguistic grid: {len(grid_centers)} rules  '
      f'({N_LEVELS}^{N_INPUTS} = {N_LEVELS**N_INPUTS})')
print(f'Centre range: {grid_centers.min():.3f} – {grid_centers.max():.3f}  '
      f'Sigma: {grid_sigmas[0,0]:.3f}')


In [ ]:
# Cell 6 – Source FNN Training  (Grid-81 initialisation)
# ──────────────────────────────────────────────────────────────────────────
# Starts with 81 linguistic-grid rules.
# Both growing and pruning (Eqs. 10-14) are active throughout training.

def train_source_fnn(Xs, ys,
                     n_iter=300, lr=0.1,
                     N_win=100, K_max=100, K_min=2, seed=42,
                     init_centers=None, init_sigmas=None):
    np.random.seed(seed)
    P = Xs.shape[1]

    if init_centers is not None:
        K0   = len(init_centers)
        fnn  = RBFFNN(P, K0, lr)
        fnn.C = init_centers.copy()
        fnn.S = init_sigmas.copy() if init_sigmas is not None                 else np.full((K0, P), LEVEL_SIG)
        fnn.W = np.random.randn(K0) * 0.1
    else:
        K0  = 1
        fnn = RBFFNN(P, K0, lr)
        fnn.C = Xs[:K0].copy()

    rule_hist = [fnn.K]
    rmse_hist = []
    U_buf     = np.zeros((N_win, fnn.K))
    y_buf     = np.zeros(N_win)
    t_global  = 0

    for ep in range(n_iter):
        sq_errs = []
        for x, yd in zip(Xs, ys):
            y, u, v = fnn.forward(x)
            sq_errs.append((y - yd) ** 2)

            ti = t_global % N_win
            if U_buf.shape[1] != fnn.K:
                U_buf = np.zeros((N_win, fnn.K))
            U_buf[ti] = u
            y_buf[ti] = y
            t_global += 1

            fnn.update(x, y, yd, u, v)

            if t_global >= N_win and t_global % N_win == 0:
                R, M, C = compute_indices(U_buf, y_buf)
                gi = int(np.argmin(R))
                pi = int(np.argmax(R))

                # Growing criterion (Eq. 13)
                if (gi == int(np.argmax(M)) == int(np.argmax(C))
                        and fnn.K < K_max):
                    fnn.add_rule(x, gi)
                    U_buf    = np.zeros((N_win, fnn.K))
                    t_global = 0

                # Pruning criterion (Eq. 14)
                elif (pi == int(np.argmin(M)) == int(np.argmin(C))
                        and fnn.K > K_min):
                    fnn.del_rule(pi)
                    U_buf    = np.zeros((N_win, fnn.K))
                    t_global = 0

        rmse_hist.append(np.sqrt(np.mean(sq_errs)))
        rule_hist.append(fnn.K)

    return fnn, rule_hist, rmse_hist

print('Source FNN (Grid-81 init, full grow+prune) training function defined.')


In [ ]:
# Cell 7 – DK-SOFNN Target Training  (Grid-81 initialisation)
# ──────────────────────────────────────────────────────────────────────────
# Target FNN starts with 81 linguistic-grid rules.
# DK-SOFNN grow / prune / replace (Eqs. 17-30) runs throughout.

def train_dk_sofnn(Xtt, ytt, src_fnn, Xs, ys,
                   n_iter=300, lr=0.01,
                   N_win=10, H=10, K_max=100, K_min=2, seed=42,
                   init_centers=None, init_sigmas=None):
    np.random.seed(seed)
    P = Xtt.shape[1]

    if init_centers is not None:
        K0   = len(init_centers)
        fnn  = RBFFNN(P, K0, lr)
        fnn.C = init_centers.copy()
        fnn.S = init_sigmas.copy() if init_sigmas is not None                 else np.full((K0, P), LEVEL_SIG)
        fnn.W = np.random.randn(K0) * 0.1
    else:
        fnn = src_fnn.clone()
        fnn.lr = lr

    # ── source reference indices (baseline thresholds) ──────────────────
    U_src = np.array([src_fnn.forward(x)[1] for x in Xs])
    y_src = np.array([src_fnn.forward(x)[0] for x in Xs])
    R_s, M_s, C_s = compute_indices(U_src, y_src)
    Rbar_s, Mbar_s, Cbar_s = R_s.mean(), M_s.mean(), C_s.mean()

    alphas = np.linspace(0.50, 1.00, H)
    betas  = 1.0 - alphas

    rule_hist = [fnn.K]
    rmse_hist = []
    U_buf     = np.zeros((N_win, fnn.K))
    y_buf     = np.zeros(N_win)
    t_global  = 0

    for ep in range(n_iter):
        sq_errs = []
        for t, (x, yd) in enumerate(zip(Xtt, ytt)):
            y, u, v = fnn.forward(x)
            sq_errs.append((y - yd) ** 2)

            ti = t_global % N_win
            if U_buf.shape[1] != fnn.K:
                U_buf = np.zeros((N_win, fnn.K))
            U_buf[ti] = u
            y_buf[ti] = y
            t_global += 1

            # ── source knowledge distillation tensors ─────────────────────
            Ks, Kt = src_fnn.K, fnn.K
            n  = min(Ks, Kt)
            Ws = np.zeros(Kt);          Ws[:n] = src_fnn.W[:n]
            Cs = np.zeros((Kt, P));     Cs[:n] = src_fnn.C[:n]
            Ss = np.full((Kt, P), 0.3); Ss[:n] = src_fnn.S[:n]

            # ── H-step look-ahead best update (Eqs. 24-30) ───────────────
            W0, C0, S0 = fnn.W.copy(), fnn.C.copy(), fnn.S.copy()
            bW, bC, bS = W0, C0, S0
            best_score  = np.inf

            for h in range(H):
                fnn.W, fnn.C, fnn.S = W0.copy(), C0.copy(), S0.copy()
                fnn.update(x, y, yd, u, v,
                           alpha=alphas[h], beta=betas[h],
                           Ws=Ws, Cs=Cs, Ss=Ss)
                if t + 1 < len(Xtt):
                    y_next, _, _ = fnn.forward(Xtt[t + 1])
                    score = (y_next - ytt[t + 1]) ** 2
                else:
                    score = (fnn.forward(x)[0] - yd) ** 2

                if score < best_score:
                    best_score = score
                    bW, bC, bS = fnn.W.copy(), fnn.C.copy(), fnn.S.copy()

            fnn.W, fnn.C, fnn.S = bW, bC, bS

            # ── structure adjustment (Eqs. 17-23) ────────────────────────
            if t_global >= N_win and t_global % N_win == 0:
                R_t, M_t, C_t = compute_indices(U_buf, y_buf)
                Rbar_t = R_t.mean()
                Mbar_t = M_t.mean()
                Cbar_t = C_t.mean()

                if (Rbar_t <= Rbar_s and Mbar_t >= Mbar_s and
                        Cbar_t <= Cbar_s and fnn.K < K_max):
                    l = int(np.argmax(C_t))
                    fnn.add_rule(x, l)
                    U_buf    = np.zeros((N_win, fnn.K))
                    t_global = 0

                elif fnn.K > K_min:
                    prune_l = -1
                    cond_A  = (Rbar_t >= Rbar_s and Cbar_t >= Cbar_s)
                    cond_B  = (Mbar_t <= Mbar_s and Cbar_t >= Cbar_s)

                    if cond_A and cond_B:
                        prune_l = int(np.argmax(R_t - M_t - C_t))
                    elif cond_A:
                        prune_l = int(np.argmin(M_t)) if Mbar_t <= Mbar_s                                   else int(np.argmax(R_t))
                    elif cond_B:
                        prune_l = int(np.argmin(M_t)) if Rbar_t <= Rbar_s                                   else int(np.argmax(R_t - M_t))

                    if prune_l >= 0:
                        fnn.del_rule(prune_l)
                        U_buf    = np.zeros((N_win, fnn.K))
                        t_global = 0

                elif ((Rbar_t >= Rbar_s or Mbar_t <= Mbar_s) and
                      Cbar_t <= Cbar_s and src_fnn.K > 0):
                    score_t = -R_t + M_t + C_t
                    l_worst = int(np.argmin(score_t))
                    z_best  = int(np.argmax(np.abs(src_fnn.W)))
                    fnn.replace_rule(l_worst,
                                     src_fnn.C[z_best],
                                     src_fnn.S[z_best],
                                     src_fnn.W[z_best])

        rmse_hist.append(np.sqrt(np.mean(sq_errs)))
        rule_hist.append(fnn.K)

    return fnn, rule_hist, rmse_hist

print('DK-SOFNN Target (Grid-81 init, full grow+prune+replace) training function defined.')


In [ ]:
# Cell 8 – Evaluation Metrics (Eqs. 42-44, unchanged)
def rmse(y_pred, y_true):
    return float(np.sqrt(np.mean((y_pred - y_true)**2)))

def smape(y_pred, y_true):
    num = 2 * np.abs(y_pred - y_true)
    den = np.abs(y_pred) + np.abs(y_true) + EPS_IDX
    return float(np.mean(num / den))

def mase(y_pred, y_true):
    scale = np.mean(np.abs(y_true[1:] - y_true[:-1])) + EPS_IDX
    return float(np.mean(np.abs(y_pred - y_true)) / scale)

print('Metrics defined.')


In [ ]:
# Cell 9 – Training: Grid-81 Source FNN → Grid-81 Target DK-SOFNN
# ──────────────────────────────────────────────────────────────────
SEED = 42

# ── Source FNN ─────────────────────────────────────────────────────────
print('Training Source FNN (K0=81, 1500 samples, 300 epochs) …')
src_fnn, rule_src, rmse_src = train_source_fnn(
    DS['Xs'], DS['ys'],
    n_iter=300, lr=0.1, N_win=100, K_max=100, K_min=2, seed=SEED,
    init_centers=grid_centers, init_sigmas=grid_sigmas)
print(f'  Source FNN: K0=81  →  final K = {src_fnn.K} rules  '
      f'RMSE = {rmse_src[-1]:.4f}')

# ── Target DK-SOFNN ────────────────────────────────────────────────────
print('Training Target DK-SOFNN (K0=81, 100 samples, 300 epochs) …')
tgt_fnn, rule_tgt, rmse_tgt = train_dk_sofnn(
    DS['Xtt'], DS['ytt'], src_fnn, DS['Xs'], DS['ys'],
    n_iter=300, lr=0.01, N_win=10, H=10, K_max=100, K_min=2, seed=SEED,
    init_centers=grid_centers, init_sigmas=grid_sigmas)
print(f'  Target FNN: K0=81  →  final K = {tgt_fnn.K} rules  '
      f'RMSE = {rmse_tgt[-1]:.4f}')

# ── Test ───────────────────────────────────────────────────────────────
pred = tgt_fnn.predict(DS['Xte'])
y_te = DS['yte']
print(f'\nTest RMSE (normalised) : {rmse(pred, y_te):.4f}')
print(f'Test sMAPE             : {smape(pred, y_te):.4f}')


In [ ]:
# Cell 10 – Source FNN Rule Count Evolution
fig, ax = plt.subplots(figsize=(9, 4))

epochs_src = np.arange(len(rule_src))
ax.plot(epochs_src, rule_src, color='steelblue', linewidth=2,
        marker='o', markevery=max(1, len(epochs_src)//20), markersize=4)
ax.axhline(y=81, color='grey', linestyle='--', linewidth=1, alpha=0.7,
           label='Initial K = 81')
ax.axhline(y=rule_src[-1], color='steelblue', linestyle=':', alpha=0.6)
ax.yaxis.set_major_locator(MaxNLocator(integer=True))
ax.set_xlabel('Epoch')
ax.set_ylabel('Number of Fuzzy Rules')
ax.set_title('Source FNN — Rule Count Evolution\n'
             f'Starts at K=81 (Grid), converges to K={rule_src[-1]}  '
             f'(source data: 1500 samples, 300 epochs)',
             fontweight='bold')
ax.annotate(f'Final: {rule_src[-1]} rules',
            xy=(epochs_src[-1], rule_src[-1]),
            xytext=(-90, 12), textcoords='offset points',
            fontsize=9, color='steelblue',
            arrowprops=dict(arrowstyle='->', color='steelblue'))
ax.legend(fontsize=9)
ax.grid(True)
plt.tight_layout()
plt.savefig('fig_source_rule_count.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_source_rule_count.png')


In [ ]:
# Cell 11 – Target DK-SOFNN Rule Count Evolution
fig, ax = plt.subplots(figsize=(9, 4))

epochs_tgt = np.arange(len(rule_tgt))
ax.plot(epochs_tgt, rule_tgt, color='darkorange', linewidth=2,
        marker='s', markevery=max(1, len(epochs_tgt)//20), markersize=4)
ax.axhline(y=81, color='grey', linestyle='--', linewidth=1, alpha=0.7,
           label='Initial K = 81')
ax.axhline(y=rule_tgt[-1], color='darkorange', linestyle=':', alpha=0.6)
ax.yaxis.set_major_locator(MaxNLocator(integer=True))
ax.set_xlabel('Epoch')
ax.set_ylabel('Number of Fuzzy Rules')
ax.set_title('Target DK-SOFNN — Rule Count Evolution\n'
             f'Starts at K=81 (Grid), converges to K={rule_tgt[-1]}  '
             f'(target-train: 100 samples, 300 epochs)',
             fontweight='bold')
ax.annotate(f'Final: {rule_tgt[-1]} rules',
            xy=(epochs_tgt[-1], rule_tgt[-1]),
            xytext=(-90, 12), textcoords='offset points',
            fontsize=9, color='darkorange',
            arrowprops=dict(arrowstyle='->', color='darkorange'))
ax.legend(fontsize=9)
ax.grid(True)
plt.tight_layout()
plt.savefig('fig_target_rule_count.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_target_rule_count.png')
